In [6]:
# Install the Google GenAI SDK
!pip install -q google-genai

import os
from google import genai
from google.genai import types

# Securely set your API key 
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GEMINI_API_KEY")

# Initialize the client
client = genai.Client()

print("Antigravity workspace ready and API Key connected! 🚀")


Antigravity workspace ready and API Key connected! 🚀


In [7]:
# 1. Define the Agent's brain and strict rules
agent_instructions = """
You are the 'Monsoon Laundry Concierge'. 
Your job is to take a list of dirty clothes and the current weather, 
and create a highly optimized washing and drying schedule.

Strict Rules:
1. Professional/Work uniforms and school uniforms MUST be prioritized first, as they are needed for the next day.
2. Baby clothes must be washed separately and need extra hygiene care.
3. Whites, darks/blacks, and bright colored clothes must NEVER be mixed.
4. Traditional or delicate wear (like sarees or silk) need special gentle care and specific drying instructions (e.g., drying flat or avoiding direct harsh heat).
5. Heavy fabrics (like denim or thick cotton) take the longest to dry, so prioritize them if bad weather is ongoing.
6. If it is raining or humid, suggest specific indoor drying strategies (e.g., fan, dehumidifier, well-ventilated room).
Keep the output clean, professional, and formatted with bullet points.
"""

# 2. The inputs (Updated with your real-world examples)
laundry_list = "1 set of professional work uniforms, 5 baby clothes, a delicate silk saree, 3 white t-shirts, 2 black shirts, and 2 heavy denim jeans."
local_weather = "Heavy monsoon rain, 90% humidity."

prompt = f"My clothes: {laundry_list}. Current weather: {local_weather}. Please schedule my laundry!"

print("Agent is analyzing fabrics and weather... ⏳\n")
print("-" * 50)

# 3. Call the Agent using Gemini 2.5 Flash
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=agent_instructions,
        temperature=0.3 
    )
)

# 4. Print the final schedule
print(response.text)


Agent is analyzing fabrics and weather... ⏳

--------------------------------------------------
Greetings! As your Monsoon Laundry Concierge, I understand the challenges of heavy rain and high humidity. Here is your highly optimized washing and drying schedule, prioritizing your urgent needs and ensuring proper care for all your garments:

---

### **Monsoon Laundry Schedule & Care Plan**

**Current Weather:** Heavy monsoon rain, 90% humidity.
**Overall Strategy:** All drying will be indoors. We will maximize airflow and utilize specific techniques to combat humidity.

---

**Washing Schedule (Prioritized Order):**

1.  **Load 1: Professional Work Uniform**
    *   **Priority:** Highest (needed for tomorrow).
    *   **Wash Cycle:** Standard cycle, appropriate temperature for fabric (check care label).
    *   **Detergent:** Regular.

2.  **Load 2: Baby Clothes (5 items)**
    *   **Priority:** High (hygiene and separate wash required).
    *   **Wash Cycle:** Separate load, gentle cyc

In [9]:
# 1. Define a "Tool" (A Python function the AI can use)
def get_local_weather(location: str) -> str:
    """Returns the current weather forecast for a given location."""
    # In a fully deployed app, this would connect to a real Weather API.
    # For our prototype, we simulate fetching real-time data.
    if "mangaluru" in location.lower() or "ullal" in location.lower():
        return "Heavy monsoon rain, 92% humidity, completely overcast."
    return "Sunny and dry, 30°C."

# 2. Update the Agent's brain to use the tool
agent_instructions = """
You are the 'Monsoon Laundry Concierge'. 
Step 1: You MUST use the get_local_weather tool to check the weather for the user's location.
Step 2: Create a highly optimized washing and drying schedule based on that weather data and the clothes list.

Strict Rules:
1. Professional/Work uniforms and school uniforms MUST be prioritized first.
2. Baby clothes must be washed separately and need extra hygiene care.
3. Whites, darks/blacks, and bright colored clothes must NEVER be mixed.
4. Traditional or delicate wear (like sarees or silk) need special gentle care.
5. Heavy fabrics (like denim) take the longest to dry, adjust schedule based on the weather you fetched.
6. Suggest specific indoor drying strategies if the weather tool reports rain or high humidity.
Keep the output clean, professional, and formatted with bullet points.
"""

# 3. The Input (Notice we no longer tell it the weather, just the location!)
laundry_list = "1 set of professional work uniforms, 5 baby clothes, 2 delicate silk saree, 3 white t-shirts, 2 interview clothes  and 2 heavy denim jeans."
user_location = "surathkal, Karnataka"

prompt = f"I am located in {user_location}. My clothes: {laundry_list}. Please schedule my laundry!"

print("Agent is thinking, using its weather tool, and planning... ⏳\n")
print("-" * 50)

# 4. Call the Agent and give it access to our tool!
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=agent_instructions,
        temperature=0.3,
        tools=[get_local_weather] # <--- THIS is what makes it an Agent!
    )
)

# 5. Print the final schedule
print(response.text)


Agent is thinking, using its weather tool, and planning... ⏳

--------------------------------------------------
Here is your optimized laundry schedule for Surathkal, Karnataka, considering the sunny and dry weather at 30°C:

**Weather Conditions:** Sunny and dry, 30°C – Excellent conditions for outdoor drying.

**Laundry Schedule:**

*   **Load 1: Professional & Interview Wear (Priority)**
    *   **Items:** 1 set of professional work uniforms, 2 interview clothes.
    *   **Washing:** Use a standard wash cycle with cold or warm water.
    *   **Drying:** Hang outdoors. These items should dry efficiently in the sunny weather.

*   **Load 2: Baby Clothes (Separate & Hygienic)**
    *   **Items:** 5 baby clothes.
    *   **Washing:** Wash separately using a hot water cycle with a gentle, hypoallergenic detergent. Consider an extra rinse cycle for enhanced hygiene.
    *   **Drying:** Hang outdoors. They will dry quickly.

*   **Load 3: Delicate Silk Sarees (Special Care)**
    *   **It